<a href="https://colab.research.google.com/github/KirillVidov/MailSorter2/blob/main/DiplomCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ИМПОРТЫ**

In [ ]:
import imaplib
import email
from email.header import decode_header
import pandas as pd
import getpass
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import sys
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import shutil

# **ДОБАВЛЕНИЕ ДАННЫХ**

In [ ]:
EMAIL_USER = "kirillvidov132@gmail.com"  # ВАШ ЛОГИН
EMAIL_PASS = "fdjv dhng hzkn eame"  # ВАШ ПАРОЛЬ ПРИЛОЖЕНИЯ
IMAP_SERVER = "imap.gmail.com"  # Для Gmail. (imap.yandex.ru / imap.mail.ru)
NUM_EMAILS = 200  # Сколько последних писем скачать

In [ ]:
def clean_text(text):
    """Убирает лишние пробелы и переносы строк"""
    if text:
        return " ".join(text.split())
    return ""

In [ ]:
def decode_str(header_value):
    """Декодирует тему письма (убирает кракозябры =?UTF-8?...)"""
    if not header_value:
        return ""
    decoded_list = decode_header(header_value)
    default_charset = 'utf-8'
    text_parts = []

    for decoded_bytes, charset in decoded_list:
        if isinstance(decoded_bytes, bytes):
            try:
                part = decoded_bytes.decode(charset or default_charset)
            except (LookupError, UnicodeDecodeError):
                part = decoded_bytes.decode(default_charset, errors='replace')
            text_parts.append(part)
        else:
            text_parts.append(str(decoded_bytes))

    return "".join(text_parts)


In [ ]:
def get_body(msg):
    """Извлекает чистый текст из письма"""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))

            # Ищем текстовую часть, игнорируем вложения
            if content_type == "text/plain" and "attachment" not in content_disposition:
                try:
                    return part.get_payload(decode=True).decode()
                except:
                    pass
    else:
        # Если письмо не составное, просто берем текст
        try:
            return msg.get_payload(decode=True).decode()
        except:
            pass
    return ""

In [ ]:
def main():
    print("Подключаемся к серверу...")
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)

    try:
        mail.login(EMAIL_USER, EMAIL_PASS)
        print("Успешный вход!")
    except Exception as e:
        print(f"Ошибка входа: {e}")
        print("Совет: Проверьте, включили ли вы 'Пароль приложения' в настройках почты.")
        return

    mail.select("inbox")

    # Поиск писем (ALL - все, UNSEEN - непрочитанные)
    status, messages = mail.search(None, "ALL")
    email_ids = messages[0].split()

    # Берем последние N писем (идем с конца)
    latest_email_ids = email_ids[-NUM_EMAILS:]

    data_list = []

    print(f"Начинаем скачивание {len(latest_email_ids)} писем...")

    for i, e_id in enumerate(reversed(latest_email_ids)):
        # Скачиваем письмо
        res, msg_data = mail.fetch(e_id, "(RFC822)")
        for response_part in msg_data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])

                # Получаем тему и отправителя
                subject = decode_str(msg["Subject"])
                sender = decode_str(msg["From"])

                # Получаем текст
                body = get_body(msg)

                # Очищаем текст (убираем HTML теги, если они просочились, можно доработать)
                # Для простоты пока просто берем text/plain

                if body:
                    data_list.append({
                        "Subject": subject,
                        "From": sender,
                        "Body": clean_text(body),
                        "Category": ""  # Пустая колонка для вашей разметки
                    })

        if i % 20 == 0:
            print(f"Обработано {i} писем...")

    # Сохраняем в CSV
    df = pd.DataFrame(data_list)
    filename = "my_emails.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig')  # utf-8-sig чтобы Excel открыл кириллицу

    print(f"\nГотово! Скачано {len(df)} писем.")
    print(f"Откройте файл {filename} и заполните колонку 'Category'.")

In [ ]:
main()

# **МОДЕЛЬ**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Настройки
MODEL_NAME = 'bert-base-multilingual-cased'

# Чтение файла
try:
    df = pd.read_csv('labeled_emails.csv')
except:
    df = pd.read_csv('labeled_emails.csv', encoding='latin1')

# Чистка
df = df.dropna(subset=['Body', 'Category'])
df['Category'] = df['Category'].astype(int)

# Объединяем тему и текст
df['full_text'] = df['Subject'].astype(str) + " " + df['From'].astype(str) + " " + df['Body'].astype(str)
df['full_text'] = df['full_text'].apply(lambda x: x[:512]) # Обрезаем слишком длинные

# Показываем пример данных (чтобы убедиться, что всё ок)
print(f"Всего строк: {len(df)}")
df[['Category', 'full_text']].head()

In [ ]:
print("Загружаем токенизатор...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

input_ids = []
attention_masks = []

print("Превращаем текст в цифры...")
for text in df['full_text'].values:
    encoded_dict = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,           # Чуть уменьшил для скорости
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(df['Category'].values)

print("Готово! Данные готовы к обучению.")

In [ ]:
# Разделение на train/test
dataset = TensorDataset(input_ids, attention_masks, labels)
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
validation_dataloader = DataLoader(val_dataset, batch_size=8)

# Загрузка BERT
print("Качаем предобученную модель...")
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    output_attentions=False,
    output_hidden_states=False,
)
model.cuda() # Отправляем на видеокарту
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [ ]:
epochs = 4
print("--- НАЧАЛО ОБУЧЕНИЯ ---")

for epoch_i in range(0, epochs):
    print(f'\n======== Эпоха {epoch_i + 1} / {epochs} ========')
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_dataloader):
        # Распаковка батча
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # Шаг обучения
        model.zero_grad()
        result = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        loss = result.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"  Средняя ошибка (Loss): {avg_train_loss:.2f}")

print("\n--- ОБУЧЕНИЕ ЗАВЕРШЕНО ---")

In [ ]:
model.eval()
predictions , true_labels = [], []

for batch in validation_dataloader:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    logits = outputs.logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    predictions.extend(np.argmax(logits, axis=1).flatten())
    true_labels.extend(label_ids.flatten())

# Красивый отчет
target_names = ['Работа (0)', 'Спам (1)', 'Сервисы (2)', 'Соцсети (3)']
print(classification_report(true_labels, predictions, target_names=target_names))

# **MAIN**

In [ ]:
# Названия папок в вашей почте, куда будем кидать письма
# ВАЖНО: Создайте эти папки в почте заранее, если их нет!
FOLDERS = {
    0: "Work",  # Личное/Работа
    1: "Spam_Sort",  # Спам (лучше не называть просто Spam, чтобы не путать с системной)
    2: "Services",  # Сервисы
    3: "Socials"  # Соцсети
}

In [ ]:
def clean_text(text):
    """Очистка текста (как при обучении)"""
    if not text: return ""
    return text[:512]  # Обрезаем, чтобы не перегружать память

In [ ]:
def get_body(msg):
    """Вытаскивает текст из письма"""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                try:
                    return part.get_payload(decode=True).decode()
                except:
                    pass
    else:
        try:
            return msg.get_payload(decode=True).decode()
        except:
            pass
    return ""


In [ ]:
def decode_str(header_value):
    """Декодирует тему письма"""
    if not header_value: return ""
    decoded_list = decode_header(header_value)
    text_parts = []
    for decoded_bytes, charset in decoded_list:
        if isinstance(decoded_bytes, bytes):
            try:
                part = decoded_bytes.decode(charset or 'utf-8')
            except:
                part = decoded_bytes.decode('utf-8', errors='replace')
            text_parts.append(part)
        else:
            text_parts.append(str(decoded_bytes))
    return "".join(text_parts)

In [ ]:
def predict_category(subject, sender, body):
    """Предсказывает категорию письма"""
    # Собираем полный текст как при обучении
    full_text = str(subject) + " " + str(sender) + " " + str(body)

    inputs = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = logits.argmax().item()
    return predicted_class_id

In [ ]:
# --- ОСНОВНОЙ ЦИКЛ ---
def mail_sorter():
    print("Подключение к почте...")
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(EMAIL_USER, EMAIL_PASS)
    mail.select("inbox")

    # Ищем только непрочитанные (UNSEEN) или все (ALL)
    # Для теста лучше взять последние 5 писем
    status, messages = mail.search(None, "ALL")
    email_ids = messages[0].split()

    print(f"Найдено писем: {len(email_ids)}")
    print("Обрабатываем последние 20 писем для теста...\n")

    # Берем последние 5
    for e_id in email_ids[-20:]:
        res, msg_data = mail.fetch(e_id, "(RFC822)")
        msg = email.message_from_bytes(msg_data[0][1])

        subject = decode_str(msg["Subject"])
        sender = decode_str(msg["From"])
        body = clean_text(get_body(msg))

        # МАГИЯ НЕЙРОСЕТИ
        category_id = predict_category(subject, sender, body)
        folder_name = FOLDERS.get(category_id, "Unknown")

        print(f"📩 От: {sender}")
        print(f"   Тема: {subject}")
        print(f"   🤖 Нейросеть думает: {category_id} -> Папка: [{folder_name}]")
        print("-" * 50)

        # --- БЛОК ПЕРЕМЕЩЕНИЯ  ---
        # result = mail.copy(e_id, folder_name)
        # if result[0] == 'OK':
        #     mail.store(e_id, '+FLAGS', '\\Deleted')
        #     print(f"   Перемещено в {folder_name}")
        # else:
        #     print("   Ошибка перемещения (папка существует?)")

    # mail.expunge() # Удалить окончательно из входящих
    mail.logout()
    print("\nГотово.")

In [ ]:
mail_sorter()